In [1]:
# Install core dependencies
!pip install -q yt-dlp gdown ffmpeg-python librosa soundfile
!apt-get install -y ffmpeg

import os
import subprocess

# 1. Ingest the full video file
video_url = "https://drive.google.com/file/d/1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW/view" # [cite: 11]
# Use gdown to download (replace with actual file ID)
file_id = "1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW"
!gdown {file_id} -O full_video.mp4

# Extract a 15-second clip (e.g., 0:15 to 0:30) [cite: 19]
!ffmpeg -y -i full_video.mp4 -ss 00:00:15 -t 00:00:15 -c:v copy -c:a copy clip_15s.mp4

# Extract the reference audio for voice cloning and transcription
!ffmpeg -y -i clip_15s.mp4 -q:a 0 -map a reference_audio.wav

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.2/182.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 46.0 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
Downloading...
From (original): https://drive.google.com/uc?id=1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW
From (redirected): https://drive.google.com/uc?id=1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW&confirm=t&uuid=c6453714-4920-4fc1-9549-95147e655be4
To: /content/full_video.mp4
100% 622M/622M [00:08<00:00, 69.5MB/s]
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl -

In [22]:
import whisper

# The 'medium' model handles code-switching much better than 'small'
model = whisper.load_model("medium")

# Use task="translate" to force the mixed audio into a clean English transcript
result = model.transcribe(
    "reference_audio.wav",
    task="translate",       # <--- The fix for mixed languages
    fp16=False,
    condition_on_previous_text=False
)

clean_english_transcript = result["text"].strip()
print(f"Clean English Transcript: {clean_english_transcript}")

Clean English Transcript: so now lets see how to protect your hygiene first step is to brush your teeth before going for a day's booking and then brush your teeth in all four directions


In [23]:
!pip install -q deep-translator

from deep_translator import GoogleTranslator

# Translate the clean English text directly to Hindi
hindi_transcript = GoogleTranslator(source='en', target='hi').translate(clean_english_transcript)
print(f"Natural Hindi Translation: {hindi_transcript}")

# Save it for the Python 3.10 TTS script
with open("hindi_transcript.txt", "w") as f:
    f.write(hindi_transcript)

print("Saved clean Hindi transcript to hindi_transcript.txt.")

Natural Hindi Translation: तो अब देखते हैं कि अपनी स्वच्छता की रक्षा कैसे करें पहला कदम एक दिन की बुकिंग के लिए जाने से पहले अपने दांतों को ब्रश करना है और फिर अपने दांतों को चारों दिशाओं में ब्रश करना है।
Saved clean Hindi transcript to hindi_transcript.txt.


In [20]:
%%writefile run_tts.py
import torch
import librosa
import subprocess
import warnings
warnings.filterwarnings("ignore")

# --- PyTorch 2.6 Workaround for Coqui TTS ---
_original_load = torch.load
def _patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_load(*args, **kwargs)
torch.load = _patched_load
# --------------------------------------------

from TTS.api import TTS

# Load the translated text
with open("hindi_transcript.txt", "r") as f:
    hindi_transcript = f.read().strip()

# Initialize Voice Cloning
device = "cuda" if torch.cuda.is_available() else "cpu"
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

# Generate Audio
output_audio_path = "hindi_dub_raw.wav"
tts.tts_to_file(
    text=hindi_transcript,
    speaker_wav="reference_audio.wav",
    language="hi",
    file_path=output_audio_path
)

# Calculate duration difference
original_duration = librosa.get_duration(path="reference_audio.wav")
generated_duration = librosa.get_duration(path=output_audio_path)
speed_factor = generated_duration / original_duration

# FFmpeg atempo limits are 0.5 to 100. Chain filters if we need to slow down drastically.
if speed_factor < 0.5:
    tempo_filter = f"atempo=0.5,atempo={speed_factor/0.5}"
else:
    tempo_filter = f"atempo={speed_factor}"

# Time-Stretch to sync durations using FFmpeg instead of librosa
subprocess.run([
    "ffmpeg", "-y", "-i", output_audio_path,
    "-filter:a", tempo_filter,
    "hindi_dub_synced.wav"
], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Voice cloning and time-stretching complete. Saved as hindi_dub_synced.wav")

Overwriting run_tts.py


In [8]:
# Upgrade pip for Python 3.10
!python3.10 -m pip install --upgrade pip

# Install PyTorch explicitly using the official CUDA wheels
!python3.10 -m pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install TTS and audio libraries
!python3.10 -m pip install TTS librosa soundfile --ignore-installed blinker

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 32.7 MB/s  0:00:15
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 105.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 168.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 54.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 176.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 14.6 MB/s  0:00:23
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 58.4 MB/s  0:00:05
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 72.8 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 74.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 72.0 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 22.3 MB/s  0:00:08
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 40.7 MB

In [24]:
!python3.10 run_tts.py

 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts
 > Text splitted to sentences.
['तो अब देखते हैं कि अपनी स्वच्छता की रक्षा कैसे करें पहला कदम एक दिन की बुकिंग के लिए जाने से पहले अपने दांतों को ब्रश करना है और फिर अपने दांतों को चारों दिशाओं में ब्रश करना है।']
 > Processing time: 13.08455753326416
 > Real-time factor: 1.040576827891377
Voice cloning and time-stretching complete. Saved as hindi_dub_synced.wav


In [18]:
# Install the missing torchcodec library
!python3.10 -m pip install torchcodec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 75.6 MB/s  0:00:00


In [14]:
# Downgrade transformers to a version compatible with Coqui XTTS
!python3.10 -m pip install transformers==4.36.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 116.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 137.4 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.3.0
    Uninstalling transformers-5.3.0:
      Successfully uninstalled transformers-5.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers]


In [1]:
%cd /content/Wav2Lip

print("Starting Wav2Lip... It will pause at 0/56 for 1-2 minutes while Numba compiles. DO NOT STOP IT!")
!python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth --face ../clip_15s.mp4 --audio ../hindi_dub_synced.wav --outfile ../lipsync_unrestored.mp4

print("Wav2Lip complete! You can now run the next cell.")

/content/Wav2Lip
Starting Wav2Lip... It will pause at 0/56 for 1-2 minutes while Numba compiles. DO NOT STOP IT!
Using cuda for inference.
Reading video frames...
Number of frames available for inference: 900
(80, 1198)
Length of mel chunks: 889
  0% 0/7 [00:00<?, ?it/s]
  0% 0/56 [00:12<?, ?it/s]
Recovering from OOM error; New batch size: 8

  0% 0/112 [00:00<?, ?it/s]
  1% 1/112 [04:43<8:43:44, 283.11s/it]
  2% 2/112 [04:46<3:37:43, 118.75s/it]
  3% 3/112 [04:50<2:00:15, 66.20s/it] 
  4% 4/112 [04:53<1:14:32, 41.41s/it]
  4% 5/112 [04:57<49:32, 27.78s/it]  
  5% 6/112 [05:01<34:38, 19.61s/it]
  6% 7/112 [05:04<25:03, 14.32s/it]
  7% 8/112 [05:08<18:50, 10.87s/it]
  8% 9/112 [05:11<14:50,  8.65s/it]
  9% 10/112 [05:15<12:02,  7.08s/it]
 10% 11/112 [05:19<10:06,  6.00s/it]
 11% 12/112 [05:22<08:53,  5.33s/it]
 12% 13/112 [05:26<08:05,  4.90s/it]
 12% 14/112 [05:30<07:22,  4.51s/it]
 13% 15/112 [05:33<06:51,  4.24s/it]
 14% 16/112 [05:37<06:28,  4.04s/it]
 15% 17/112 [05:41<06:11,  3.92

In [4]:
%cd /content
import os

# Safety Check: Ensure Wav2Lip actually finished
vid_path = 'lipsync_unrestored.mp4'
if not os.path.exists(vid_path) or os.path.getsize(vid_path) < 1000:
    print("🚨 ERROR: lipsync_unrestored.mp4 is missing or completely empty!")
    print("🚨 You MUST go back and run the Wav2Lip cell until the progress bar reaches 100%.")
else:
    print("✅ Wav2Lip video verified! Starting production-grade frame pipeline...")

    # 1. Clean up old folders and extract frames from the Wav2Lip video
    !rm -rf extracted_frames restored_frames
    !mkdir -p extracted_frames
    print("Extracting frames...")
    !ffmpeg -y -i lipsync_unrestored.mp4 -qscale:v 1 -qmin 1 -qmax 1 extracted_frames/frame_%04d.jpg -hide_banner -loglevel error

    # 2. Run GFPGAN on the extracted images instead of the video file
    %cd /content/GFPGAN
    print("Running GFPGAN on individual frames (This will take a minute)...")
    !python inference_gfpgan.py -i /content/extracted_frames -o /content/restored_frames -v 1.4 -s 2 --bg_upsampler realesrgan

    # 3. Restitch the restored frames with your naturally-paced Hindi audio
    %cd /content
    print("Restitching frames and syncing audio...")
    !ffmpeg -y -framerate 25 -i restored_frames/restored_imgs/frame_%04d.jpg -i hindi_dub_synced.wav -c:v libx264 -pix_fmt yuv420p -c:a aac -strict experimental final_supernan_dub.mp4 -hide_banner -loglevel error

    print("🎉 MASTERPIECE COMPLETE! Download 'final_supernan_dub.mp4' from your main /content/ folder.")

Streaming output truncated to the last 5000 lines.
	Tile 12/15
	Tile 13/15
	Tile 14/15
	Tile 15/15
Processing frame_0578.jpg ...
	Tile 1/15
	Tile 2/15
	Tile 3/15
	Tile 4/15
	Tile 5/15
	Tile 6/15
	Tile 7/15
	Tile 8/15
	Tile 9/15
	Tile 10/15
	Tile 11/15
	Tile 12/15
	Tile 13/15
	Tile 14/15
	Tile 15/15
Processing frame_0579.jpg ...
	Tile 1/15
	Tile 2/15
	Tile 3/15
	Tile 4/15
	Tile 5/15
	Tile 6/15
	Tile 7/15
	Tile 8/15
	Tile 9/15
	Tile 10/15
	Tile 11/15
	Tile 12/15
	Tile 13/15
	Tile 14/15
	Tile 15/15
Processing frame_0580.jpg ...
	Tile 1/15
	Tile 2/15
	Tile 3/15
	Tile 4/15
	Tile 5/15
	Tile 6/15
	Tile 7/15
	Tile 8/15
	Tile 9/15
	Tile 10/15
	Tile 11/15
	Tile 12/15
	Tile 13/15
	Tile 14/15
	Tile 15/15
Processing frame_0581.jpg ...
	Tile 1/15
	Tile 2/15
	Tile 3/15
	Tile 4/15
	Tile 5/15
	Tile 6/15
	Tile 7/15
	Tile 8/15
	Tile 9/15
	Tile 10/15
	Tile 11/15
	Tile 12/15
	Tile 13/15
	Tile 14/15
	Tile 15/15
Processing frame_0582.jpg ...
	Tile 1/15
	Tile 2/15
	Tile 3/15
	Tile 4/15
	Tile 5/15
	Tile 6/15
	

In [5]:
%cd /content

print("Restitching frames at the CORRECT 60 FPS and syncing audio...")
!ffmpeg -y -framerate 60 -i restored_frames/restored_imgs/frame_%04d.jpg -i hindi_dub_synced.wav -c:v libx264 -pix_fmt yuv420p -c:a aac -strict experimental final_supernan_dub_fixed.mp4 -hide_banner -loglevel error

print("🎉 PERFECT SYNC COMPLETE! Download 'final_supernan_dub_fixed.mp4' from your main /content/ folder.")

/content
Restitching frames at the CORRECT 60 FPS and syncing audio...
🎉 PERFECT SYNC COMPLETE! Download 'final_supernan_dub_fixed.mp4' from your main /content/ folder.


In [6]:
%cd /content
import os

print("1. Isolating the speaker to ignore the TV screen...")
# Crop exactly the right half of the video where the speaker is standing
!ffmpeg -y -i clip_15s.mp4 -filter:v "crop=iw/2:ih:iw/2:0" right_half.mp4 -hide_banner -loglevel error

print("2. Running Wav2Lip ONLY on the isolated speaker...")
%cd /content/Wav2Lip
!python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth --face ../right_half.mp4 --audio ../hindi_dub_synced.wav --outfile ../right_half_synced.mp4

print("3. Stitching the synchronized speaker seamlessly back into the room...")
%cd /content
# Overlay the talking right half back onto the original left half
!ffmpeg -y -i clip_15s.mp4 -i right_half_synced.mp4 -filter_complex "[0:v][1:v]overlay=W/2:0" -c:a copy full_room_synced.mp4 -hide_banner -loglevel error

print("4. Extracting frames for face restoration...")
!rm -rf extracted_frames restored_frames
!mkdir -p extracted_frames
!ffmpeg -y -i full_room_synced.mp4 -qscale:v 1 -qmin 1 -qmax 1 extracted_frames/frame_%04d.jpg -hide_banner -loglevel error

print("5. Running GFPGAN to sharpen the face...")
%cd /content/GFPGAN
!python inference_gfpgan.py -i /content/extracted_frames -o /content/restored_frames -v 1.4 -s 2 --bg_upsampler realesrgan

print("6. Compiling final 60 FPS masterpiece...")
%cd /content
!ffmpeg -y -framerate 60 -i restored_frames/restored_imgs/frame_%04d.jpg -i hindi_dub_synced.wav -c:v libx264 -pix_fmt yuv420p -c:a aac -strict experimental final_supernan_dub_flawless.mp4 -hide_banner -loglevel error

print("🎉 FLAWLESS SYNC COMPLETE! Download 'final_supernan_dub_flawless.mp4'")

Streaming output truncated to the last 5000 lines.
	Tile 12/15
	Tile 13/15
	Tile 14/15
	Tile 15/15
Processing frame_0589.jpg ...
	Tile 1/15
	Tile 2/15
	Tile 3/15
	Tile 4/15
	Tile 5/15
	Tile 6/15
	Tile 7/15
	Tile 8/15
	Tile 9/15
	Tile 10/15
	Tile 11/15
	Tile 12/15
	Tile 13/15
	Tile 14/15
	Tile 15/15
Processing frame_0590.jpg ...
	Tile 1/15
	Tile 2/15
	Tile 3/15
	Tile 4/15
	Tile 5/15
	Tile 6/15
	Tile 7/15
	Tile 8/15
	Tile 9/15
	Tile 10/15
	Tile 11/15
	Tile 12/15
	Tile 13/15
	Tile 14/15
	Tile 15/15
Processing frame_0591.jpg ...
	Tile 1/15
	Tile 2/15
	Tile 3/15
	Tile 4/15
	Tile 5/15
	Tile 6/15
	Tile 7/15
	Tile 8/15
	Tile 9/15
	Tile 10/15
	Tile 11/15
	Tile 12/15
	Tile 13/15
	Tile 14/15
	Tile 15/15
Processing frame_0592.jpg ...
	Tile 1/15
	Tile 2/15
	Tile 3/15
	Tile 4/15
	Tile 5/15
	Tile 6/15
	Tile 7/15
	Tile 8/15
	Tile 9/15
	Tile 10/15
	Tile 11/15
	Tile 12/15
	Tile 13/15
	Tile 14/15
	Tile 15/15
Processing frame_0593.jpg ...
	Tile 1/15
	Tile 2/15
	Tile 3/15
	Tile 4/15
	Tile 5/15
	Tile 6/15
	